# 環境変数

In [ ]:
import os
os.environ["ERG_DATA_DIR"] = "/mnt/j/observation_data/"

# 3dfluxデータを時間軸に焼き直す

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr

pt.del_data('*')

time_range_full = ['2017-11-15/16:00:00', '2017-11-15/17:00:00']
psp.erg.lepi(time_range_full, datatype='3dflux', get_support_data=True, no_update=True, version='v03_00')

# 解析対象の狭い時間窓
time_range = ['2017-11-15/16:10:00', '2017-11-15/16:25:00']

# flux3d本体 (dims = time, v1(energy), v2(channel), v3(phase))  # [#/cm2/sr/sec/keV]
flux3d = pt.data_quants['erg_lepi_l2_3dflux_FPDU'].sel(
    time=slice(*time_range)
)

# 各軸の座標を取り出しておく
time_ax     = flux3d.time.values        # shape = (T,)
energy_ax   = flux3d.v1.values          # (E,)
channel_ax  = flux3d.v2.values          # (C,)
spin_ax     = flux3d.v3.values          # (S,)

time_num, energy_num, channel_num, spin_num = len(time_ax), len(energy_ax), len(channel_ax), len(spin_ax)   # T, E, C, S

# 1 spin time = 8 sec
# 1 spin phase time = 0.5 sec
# 1 energy time step = 15625 μsec
spin_offset_ns      = np.arange(spin_num, dtype='timedelta64[ns]') * 500_000_000        # 0.5 sec = 500,000,000 nsec
energy_offset_ns    = (np.arange(energy_num, dtype='int64') * 15_625_000 + 7_812_500).astype('timedelta64[ns]')

offset_ns           = energy_offset_ns[:, None] + spin_offset_ns    # (E, 1) + (S) -> (E, S)

flux_E_TS_C = flux3d.transpose('v1_dim', 'time', 'v3_dim', 'v2_dim').values.reshape(energy_num, time_num*spin_num, channel_num)

# xarray.DataArrayをエネルギーごとに生成
flux_data_arrays = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1) # ((T, 1) + (1, S) -> (T, S)).reshape(-1) -> (TxS,)

    flux_data_arrays[energy_i] = xr.DataArray(
        flux_E_TS_C[energy_i],
        dims=['time', 'channel'],
        coords={
            'time':         time_flat,
            'channel':      channel_ax,
            'energy_keV':   energy_ax[energy_i]
        },
        attrs=flux3d.attrs,
        name=f'flux_energy_{energy_i}'
    )

print(flux_data_arrays)

fidu_angle_dict     = pt.data_quants['erg_lepi_l2_3dflux_FIDU_Angle_sga']
fidu_angle  = fidu_angle_dict['data'].astype(float)     # shape (2, 3, 16)
AZ_deg_mid = fidu_angle[0, 1, :channel_num]      # shape (C,)
# AZ_deg_mid =  [ 78.75  56.25  33.75  11.25 -11.25 -33.75 -56.25 -78.75]

theta_sga = AZ_deg_mid                  # (C,)
varphi_sga = -90. * np.ones(spin_num)   # (S,)

# (TxS, C)の2次元配列を生成
theta_sga_time = np.tile(theta_sga, (time_num*spin_num, 1))   # (TxS, C)
varphi_sga_times = np.tile(varphi_sga, (time_num, 1)).reshape(-1, 1)   # (TxS, 1)
varphi_sga_time = np.tile(varphi_sga_times, (1, channel_num))   # (TxS, C)

angle_sga_time = np.stack((theta_sga_time, varphi_sga_time), axis=2)   # (TxS, C, 2)

# theta_sga, varphi_sga -> Vx_sga, Vy_sga, Vz_sga
vector_sga_time = np.zeros((time_num*spin_num, channel_num, 3))   # (TxS, C, 3)
vector_sga_time[:, :, 0] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.cos(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 1] = np.cos(np.radians(angle_sga_time[:, :, 0])) * np.sin(np.radians(angle_sga_time[:, :, 1]))
vector_sga_time[:, :, 2] = np.sin(np.radians(angle_sga_time[:, :, 0]))

v_unit_vector_sga_energy_channel_list = {}
for energy_i in range(energy_num):
    time_flat = (time_ax[:, None] + offset_ns[energy_i][None, :]).reshape(-1)

    for channel_i in range(channel_num):
        v_unit_vector_sga_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            vector_sga_time[:, channel_i, :],
            dims=['time', 'xyz'],
            coords={
                'time': time_flat,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sga_{energy_i}_{channel_i}'
        )
        dot_ = (v_unit_vector_sga_energy_channel_list[energy_i, channel_i] * v_unit_vector_sga_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sga_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_sgi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        pt.store_data(f'vector_sga_{energy_i}_{channel_i}', data={'x': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].time, 'y': v_unit_vector_sga_energy_channel_list[energy_i, channel_i].values})
        # SGI座標系に変換
        psp.projects.erg.sga2sgi(name_in=f'vector_sga_{energy_i}_{channel_i}', name_out=f'vector_sgi_{energy_i}_{channel_i}')
        _data   = pt.data_quants[f'vector_sgi_{energy_i}_{channel_i}'].rename({"v_dim": "xyz"})
        _data_unit  = _data
        v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_sgi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        _dot    = (v_unit_vector_sgi_energy_channel_list[energy_i, channel_i] * v_unit_vector_sgi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_sgi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(_dot), np.nanmax(_dot), np.nanmean(_dot))

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr

v_unit_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        # dsi座標系に変換
        psp.projects.erg.sgi2dsi(name_in=f'vector_sgi_{energy_i}_{channel_i}', name_out=f'vector_dsi_{energy_i}_{channel_i}')
        _data   = pt.data_quants[f'vector_dsi_{energy_i}_{channel_i}'].rename({"v_dim": "xyz"})
        #_data_2 = (_data * _data).sum(dim='xyz')
        _data_unit  = _data #/ np.sqrt(_data_2)
        v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            _data_unit.data,
            dims=['time', 'xyz'],
            coords={
                'time': _data.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'v_unit_vector_dsi_{energy_i}_{channel_i}'
        )

for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        dot_    = (v_unit_vector_dsi_energy_channel_list[energy_i, channel_i] * v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]).sum(dim='xyz')
        print(f'v_unit_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', np.nanmin(dot_), np.nanmax(dot_), np.nanmean(dot_))

# 背景磁場ベクトルの決定

In [ ]:
import pyspedas as psp
import pytplot as pt
import xarray as xr
import numpy as np

psp.erg.mgf(trange=time_range_full, level='l2', datatype='256hz', coord='dsi', version='v03.03', no_update=True)
psp.erg.mgf(trange=time_range_full, level='l2', datatype='8sec', coord='dsi', version='v03.03', no_update=True)

B_256Hz = pt.data_quants['erg_mgf_l2_mag_256hz_dsi'].rename({"v_dim": "xyz"})
B_8sec  = pt.data_quants['erg_mgf_l2_mag_8sec_dsi'].rename({"v_dim": "xyz"})

In [ ]:
background_time_sec = 100 #[sec]

In [ ]:
B0_vector_dsi_energy_channel_list = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        B_256Hz_interp       = B_256Hz.interp(time=v_unit_vector_dsi_energy_channel_list[energy_i, channel_i].time, method='linear')
        dt_B_256Hz_interp    = (B_256Hz_interp.time[1] - B_256Hz_interp.time[0]) / np.timedelta64(1, 's')
        B_background        = B_256Hz_interp.rolling(time=int(background_time_sec/dt_B_256Hz_interp), center=True).mean('time')

        B0_vector_dsi_energy_channel_list[energy_i, channel_i] = xr.DataArray(
            B_background.data,
            dims=['time', 'xyz'],
            coords={
                'time': B_background.time,
                'xyz': ['x', 'y', 'z'],
                'channel': channel_ax[channel_i],
                'energy_keV':   energy_ax[energy_i]
            },
            name=f'B0_vector_dsi_{energy_i}_{channel_i}'
        )
        print(f'B0_vector_dsi_energy_channel_list[{energy_i}, {channel_i}] = ', B0_vector_dsi_energy_channel_list[energy_i, channel_i])

# 対象とする垂直磁場成分を取得

In [ ]:
t_B_8sec    = B_8sec.time
dt_B_8sec = (t_B_8sec[2] - t_B_8sec[1]) / np.timedelta64(1, 's')
B_background    = B_8sec.rolling(time=int(background_time_sec/dt_B_8sec), center=True).mean('time')

B_background_256Hz  = B_background.interp(time=B_256Hz.time, method='linear')
B_256Hz_perturb     = B_256Hz - B_background_256Hz
B_256Hz_perp = B_256Hz_perturb - (B_256Hz_perturb * B_background_256Hz).sum(dim='xyz') / (B_background_256Hz * B_background_256Hz).sum(dim='xyz') * B_background_256Hz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt, sosfreqz

# ------------------ フィルタ設計 ------------------
fs = 256.                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth（2-pole × 2-stage）

# 0.6 Hzから0.75 Hzの範囲のみを通すband-passフィルタ
lowcut = 0.60
highcut = 0.75

# btypeを'bandpass'に設定
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

# ------------------ インパルス応答 (変更なし) ------------------
n = 4096
delta = np.zeros(n)
delta[n//2] = 1

h = sosfiltfilt(sos, delta)
t = (np.arange(n) - n//2) / fs

# ------------------ 周波数応答 (変更なし) ------------------
w, H = sosfreqz(sos, worN=4096, fs=fs)
H_dbl = np.abs(H)**2

# ------------------ プロット (タイトルとハイライトを変更) ------------------
fig, axs = plt.subplots(2, 1, figsize=(10, 6), tight_layout=True)

# 時間領域
axs[0].plot(t, h)
axs[0].set_title('Impulse Response (Band-pass filter)')
axs[0].set_xlabel('Time [s]')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True)

# 周波数領域
axs[1].semilogx(w, 20*np.log10(H_dbl), label='|H(f)|')

# 除去帯域を半透明のグレーで示す
axs[1].axvspan(lowcut, highcut, color='gray', alpha=0.3, label=f'{lowcut:.2f}-{highcut:.2f} Hz')
axs[1].axvline(0.60, color='red', lw=1)
axs[1].axvline(0.75, color='red', lw=1)

axs[1].set_title('Magnitude Response (Band-pass filter)')
axs[1].set_xlabel('Frequency [Hz]')
axs[1].set_ylabel('Magnitude [dB]')
axs[1].set_ylim(-20, 10)
axs[1].set_xlim(3E-1, 1)
axs[1].legend()
axs[1].grid(True, which='both', ls='--')

plt.show()

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt
import os # osモジュールもインポートしておく

# フィルタパラメータ
fs = 256.                  # サンプリング周波数 [Hz]
lowcut = 0.60
highcut = 0.75
order = 4                 # フィルタの次数
window_sec = background_time_sec        # 背景磁場の移動平均窓幅 [sec]

sos = butter(N=order, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')

def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

B_256Hz_perp_bandpass = np.zeros(B_256Hz_perp.data.shape) * np.nan  # NaNで初期化
for i in range(3):
    B_256Hz_perp_bandpass[:, i] = apply_filter_segmented(B_256Hz_perp.data[:, i], sos)
da_B_256Hz_perp_bandpass = xr.DataArray(
    B_256Hz_perp_bandpass,
    dims=B_256Hz_perp.dims,
    coords=B_256Hz_perp.coords,
    name='B_256Hz_perp_bandpass'
)

da_B_256Hz_perp_bandpass_amp = np.sqrt((da_B_256Hz_perp_bandpass * da_B_256Hz_perp_bandpass).sum(dim='xyz'))

In [ ]:
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

# plot
fig, ax = plt.subplots(1, 1, figsize=(10, 4), sharex=True)
ax.plot(da_B_256Hz_perp_bandpass_amp.time, da_B_256Hz_perp_bandpass_amp.data, lw=0.5, c='k')
ax.set_ylabel('[nT]')
ax.minorticks_on()
ax.grid(True, which='both', linestyle='--', alpha=0.5)
ax.set_xlim(time_ax_range_min, time_ax_range_max)
ax.set_ylim(0, 10)
plt.tight_layout()
plt.show()

# v_unitとB0、B_perpとのなす角を求めて、pitch angleとzeta angleをfluxデータに付与

In [ ]:
def ensure_xyz_coord(da):
    if 'xyz' in da.dims and 'xyz' not in da.coords:
        da = da.assign_coords(xyz=['x','y','z'])
    return da

flux_pitch_zeta_data_list    = {}
for energy_i in range(energy_num):
    for channel_i in range(channel_num):
        v_unit  = v_unit_vector_dsi_energy_channel_list[energy_i, channel_i]
        B0 = B0_vector_dsi_energy_channel_list[energy_i, channel_i].interp(time=v_unit.time)
        Bperp = da_B_256Hz_perp_bandpass.interp(time=v_unit.time)
        Bperp = Bperp - (Bperp * B0).sum(dim='xyz') / (B0 * B0).sum(dim='xyz') * B0

        v_unit  = ensure_xyz_coord(v_unit)
        B0      = ensure_xyz_coord(B0)
        Bperp   = ensure_xyz_coord(Bperp)

        dot_vB0 = (v_unit * B0).sum(dim='xyz')
        B0_2    = (B0 * B0).sum(dim='xyz')
        alpha = np.arccos(dot_vB0 / np.sqrt(B0_2))

        v_perp = v_unit - dot_vB0 / B0_2 * B0
        cross = xr.apply_ufunc(np.cross, Bperp, v_perp,
                               input_core_dims=[['xyz'], ['xyz']],
                               output_core_dims=[['xyz']], vectorize=True)
        Bperp_2     = (Bperp * Bperp).sum(dim='xyz')
        sin_zeta    = (cross * B0).sum(dim='xyz') / np.sqrt(B0_2 * Bperp_2)
        cos_zeta    = (v_perp * Bperp).sum(dim='xyz') / np.sqrt(Bperp_2)
        zeta = np.atan2(sin_zeta, cos_zeta)
        
        # 時間を統一（zeta基準）
        t = zeta['time']
        
        # flux の time を zeta に合わせる（必要なら補間）
        flux_ch = xr.DataArray(
            flux_data_arrays[energy_i][:, channel_i],
            coords={'time': flux_data_arrays[energy_i].coords['time']},  # ここは実データのtimeに合わせる
            dims=('time',)
        ).interp(time=t)
        
        da = xr.concat(
            [
                flux_ch.rename('differential_number_flux_keV'),
                np.rad2deg(alpha).rename('pitch_angle_deg'),
                (np.rad2deg(zeta) % 360.0).rename('zeta_angle_deg')
            ],
            dim='variable'
        ).assign_coords(variable=['differential_number_flux_keV','pitch_angle_deg','zeta_angle_deg']) \
         .transpose('time','variable')
        
        flux_pitch_zeta_data_list[energy_i, channel_i] = da.assign_coords(
            channel=channel_ax[channel_i],
            energy_keV=energy_ax[energy_i],
        )
        print(flux_pitch_zeta_data_list[energy_i, channel_i])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib as mpl

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

time_ax_range_min = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max = np.datetime64('2017-11-15T16:22:00')

for energy_i in range(energy_num):
    if energy_i != 5:
        continue

    # --- 1) 全channelでvmin/vmaxを決める（>0 かつ有限のみ） ---
    flux_vals = []
    alpha_vals = []
    for channel_i in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        f = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        flux_vals.append(f.ravel())
        alpha_vals.append(alpha.ravel())
    flux_all = np.concatenate(flux_vals)
    alpha_all = np.concatenate(alpha_vals)
    mask = np.isfinite(flux_all) & (flux_all > 0) & (alpha_all > 125) & (alpha_all < 145)
    if not mask.any():
        continue
    vmin, vmax = flux_all[mask].min(), flux_all[mask].max()
    if np.log10(vmin) < np.log10(vmax) -2:
        vmin = vmax*1E-2
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    cmap = 'turbo'

    # --- 2) 描画 ---
    fig = plt.figure(figsize=(10, 12))
    ax0 = fig.add_subplot(211)
    ax1 = fig.add_subplot(212)

    for channel_i in range(channel_num):
        #if channel_i != 0:
        #    continue
        d = flux_pitch_zeta_data_list[energy_i, channel_i].sel(time=slice(time_ax_range_min, time_ax_range_max))
        t = d['time'].values
        flux = d.sel(variable='differential_number_flux_keV').values
        alpha = d.sel(variable='pitch_angle_deg').values
        zeta  = d.sel(variable='zeta_angle_deg').values

        mask = (alpha >= 125) & (alpha <= 145) & (flux > 0)
        t_mask  = t[mask]
        flux_mask   = flux[mask]
        flux_mask_ratio = flux_mask
        alpha_mask  = alpha[mask]
        zeta_mask   = zeta[mask]
        ax0.scatter(t_mask, alpha_mask, c=flux_mask_ratio, s=10, cmap=cmap, norm=norm, rasterized=True)
        ax1.scatter(t_mask, zeta_mask,  c=flux_mask_ratio, s=10, cmap=cmap, norm=norm, rasterized=True)

    # 軸体裁
    ax0.set_ylabel(r'Pitch Angle $\alpha$' + '\n[deg]')
    ax1.set_ylabel(r'Phase difference $\zeta$' + '\n[deg]')

    ax0.set_title(f'LEP-i flux (energy = {energy_ax[energy_i]:.4f} keV)')
    for ax in (ax0, ax1):
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        ax.minorticks_on()
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_xlim(time_ax_range_min, time_ax_range_max)
    ax0.set_ylim(0, 180);  ax0.set_yticks(np.arange(0, 181, 15))
    ax1.set_ylim(0, 360);  ax1.set_yticks(np.arange(0, 361, 30))

    # --- 3) カラーバーは共通norm/cmapから作る ---
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])  # 必須
    plt.colorbar(sm, ax=ax0, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')
    plt.colorbar(sm, ax=ax1, label='Differential number flux\n' + r'[$\mathrm{s}^{-1}\mathrm{cm}^{-2}\mathrm{str}^{-1}\mathrm{keV}^{-1}$]')

    plt.tight_layout()
    plt.show()


# Differential number fluxの積分

In [ ]:
import numpy as np
from scipy.spatial import Delaunay

def integrate_sinalpha_triangulation(alpha_deg, zeta_deg, f,
                                     alpha_min_deg, alpha_max_deg,
                                     zeta_min_deg,  zeta_max_deg):
    """
    Delaunay分割で ∫∫ sin(α) f dα dζ を近似（α, ζ は度で与える）。
    返り値: (I, coverage)
      I         : 積分値（α, ζ は弧度法に変換して積分）
      coverage  : 幾何学的被覆率 = (三角形面積の総和) / (範囲の箱面積) ∈ [0,1]
    """

    # ---- 入力を弧度へ ----
    a = np.deg2rad(np.asarray(alpha_deg, float))
    z = np.deg2rad(np.asarray(zeta_deg,  float)) % (2*np.pi)
    f = np.asarray(f, float)

    # ---- 範囲（弧度）を正規化：ζは [z0, z1] で z1>z0 に張り直す ----
    amin = np.deg2rad(alpha_min_deg)
    amax = np.deg2rad(alpha_max_deg)
    z0 = np.deg2rad(zeta_min_deg) % (2*np.pi)
    z1 = np.deg2rad(zeta_max_deg) % (2*np.pi)
    if z1 <= z0:
        z1 += 2*np.pi
    Lz = z1 - z0
    La = amax - amin
    if La <= 0 or Lz <= 0:
        raise ValueError("角度範囲が不正（最小<最大 になるよう指定）")

    # ζを範囲へ平行移動
    z_shift = z.copy()
    z_shift[z_shift < z0] += 2*np.pi

    # ---- 範囲内かつ有限のみ採用 ----
    m = np.isfinite(f) & (a >= amin) & (a <= amax) & (z_shift >= z0) & (z_shift <= z1)
    if m.sum() < 3:
        return np.nan, 0.0
    a, z_shift, f = a[m], z_shift[m], f[m]

    # ---- 三角形分割の準備 ----
    full_circle = np.isclose(Lz, 2*np.pi, rtol=0, atol=1e-12)

    if full_circle:
        # 全周の場合のみ周期の継ぎ目を越える三角形を作るために複製
        Z = np.concatenate([z_shift-2*np.pi, z_shift, z_shift+2*np.pi])
        A = np.concatenate([a, a, a])
        F = np.concatenate([f, f, f])
    else:
        Z, A, F = z_shift, a, f

    pts = np.column_stack([Z, A])  # 列順: [ζ, α]
    tri = Delaunay(pts)

    # ---- 積分本体 ----
    box_area = Lz * La  # 幾何学的な箱面積（弧度）
    I_sum = 0.0
    A_sum = 0.0

    for s in tri.simplices:
        P = pts[s]   # 形状 (3,2) -> [ζ, α]
        fv = F[s]

        # 三角形重心で領域内判定
        c = P.mean(axis=0)
        cz, ca = c[0], c[1]

        # full_circle のときは [z0, z0+2π] へ折り返して判定
        if full_circle:
            cz = ((cz - z0) % (2*np.pi)) + z0

        if not (z0 <= cz <= z1 and amin <= ca <= amax):
            continue

        # 複製を使った場合は，元区間外の頂点を含む三角形は除外して二重計上を防ぐ
        if full_circle and ((P[:,0] < z0).any() or (P[:,0] > z1).any()):
            continue

        # 面積（ζ–α平面）
        area = 0.5 * abs(np.linalg.det([[P[1,0]-P[0,0], P[1,1]-P[0,1]],
                                        [P[2,0]-P[0,0], P[2,1]-P[0,1]]]))

        # 重み w = sin(α)（dα dζ の面積要素に掛ける）
        w = np.sin(P[:,1])

        # ∫Δ sin(α) f dA ≈ area * mean(w*f)
        I_sum += area * np.mean(w * fv)
        A_sum += area

    if A_sum == 0:
        return np.nan, 0.0

    coverage = min(A_sum / box_area, 1.0)
    return I_sum, coverage


In [ ]:
energy_center   = energy_ax[1:]

log_center = np.log10(energy_center)
log_edges = np.zeros(len(energy_center) + 1)
log_edges[1:-1] = 0.5 * (log_center[:-1] + log_center[1:])
log_edges[0] = log_center[0] + (log_center[0] - log_edges[1])
log_edges[-1] = log_center[-1] - (log_edges[-2] - log_center[-1])

energy_grid = 10**log_edges

energy_width = np.zeros(len(energy_center))
energy_width = energy_grid[0:-1] - energy_grid[1:]

In [ ]:
proton_mass         = 1.67262192E-27    # [kg]
elementary_charge   = 1.60217663E-19    # [C]

In [ ]:
import numpy as np
from scipy.spatial import Delaunay

def build_triangulation(alpha_deg, zeta_deg, f):
    # 入力: ±30 sで集めた“全”点（範囲で絞らない）
    a = np.deg2rad(np.asarray(alpha_deg, float))
    z = np.deg2rad(np.asarray(zeta_deg,  float)) % (2*np.pi)
    f = np.asarray(f, float)
    m = np.isfinite(f)
    if m.sum() < 3:
        return None, None, None, None
    a, z, f = a[m], z[m], f[m]

    # 全周の連続性向上のため ζ を複製（補間が継ぎ目で途切れないように）
    Z = np.concatenate([z-2*np.pi, z, z+2*np.pi])
    A = np.concatenate([a, a, a])
    F = np.concatenate([f, f, f])

    tri = Delaunay(np.column_stack([Z, A]))
    return tri, Z, A, F

def integrate_sinalpha_on_rect(tri, Z, A, F,
                               alpha_min_deg, alpha_max_deg,
                               zeta_min_deg,  zeta_max_deg,
                               n_samples=20000, rng=np.random):
    if tri is None:
        return np.nan, 0.0

    # --- 角度範囲をラジアンに変換 ---
    amin = np.deg2rad(alpha_min_deg)
    amax = np.deg2rad(alpha_max_deg)
    z0   = np.deg2rad(zeta_min_deg) % (2*np.pi)
    z1   = np.deg2rad(zeta_max_deg) % (2*np.pi)
    if z1 <= z0:
        z1 += 2*np.pi

    # --- 一様ランダムサンプリング ---
    zs = z0 + (z1 - z0) * rng.random(n_samples)
    as_ = amin + (amax - amin) * rng.random(n_samples)

    # --- どの三角形に入るか判定 ---
    pts = np.column_stack([zs, as_])      # shape = (N,2)
    simplex = tri.find_simplex(pts)       # shape = (N,)
    valid = simplex >= 0
    if not np.any(valid):
        return np.nan, 0.0

    # --- バリセントリック座標による補間---
    X = pts[valid]                                  # (M,2)
    T = tri.transform[simplex[valid]]               # (M,3,2)
    delta = X - T[:,2]                               # (M,2)
    bary = np.einsum('mij,mj->mi', T[:,:2], delta)  # (M,2)
    w0 = 1 - bary[:,0] - bary[:,1]                  # (M,)
    w1 = bary[:,0]                                   # (M,)
    w2 = bary[:,1]                                   # (M,)

    verts = tri.simplices[simplex[valid]]           # (M,3)
    f_interp = w0*F[verts[:,0]] + w1*F[verts[:,1]] + w2*F[verts[:,2]]

    # --- f × sin(α) を積分 ---
    g = np.sin(as_[valid]) * f_interp
    R_area = (z1 - z0) * (amax - amin)
    I = R_area * np.nanmean(g)
    coverage = valid.mean()
    return I, coverage



In [ ]:
import numpy as np

# --- 積分パラメータ ---
win_sec = 30.0
step_sec = 4.0
t0_list = np.arange(np.datetime64('2017-11-15T16:15:00'),
                    np.datetime64('2017-11-15T16:25:00') + np.timedelta64(1,'s'),
                    np.timedelta64(int(step_sec),'s'))
t0_list_ns = t0_list.astype('datetime64[ns]')
nT = len(t0_list)

step_rad_pm = 15.
zeta_0_list = np.arange(step_rad_pm, 360. - step_rad_pm + 1E-12, 2*step_rad_pm)
zeta_bin_num = len(zeta_0_list)

alpha_min   = 125.
alpha_max   = 145.

def _gather_samples(energy_i, tmin, tmax, alpha_min, alpha_max, zeta_min, zeta_max):
    ts, al, ze, ff = [], [], [], []
    for ch in range(channel_num):
        d = flux_pitch_zeta_data_list[energy_i, ch].sel(time=slice(tmin, tmax))
        f = d.sel(variable='differential_number_flux_keV').values
        zeta = d.sel(variable='zeta_angle_deg').values
        alpha = d.sel(variable='pitch_angle_deg').values
        m = np.isfinite(f) & (alpha >= alpha_min) & (alpha <= alpha_max) & (zeta >= zeta_min) & (zeta <= zeta_max)
        if not m.any():
            continue
        ts.append(d['time'].values[m])
        al.append(d.sel(variable='pitch_angle_deg').values[m])
        ze.append(d.sel(variable='zeta_angle_deg').values[m] % 360.0)  # 0–360 に折返し
        ff.append(f[m])
    if not ts:  # データなし
        return None, None, None
    return np.concatenate(al), np.concatenate(ze), np.concatenate(ff)

N_per_energy_list      = {}
N_per_energy_cov_list  = {}

from joblib import Parallel, delayed
import xarray as xr

def compute_single_zeta(energy_i, zeta_0):
    """
    ある energy_i & zeta_0 に対して
    t0_list の全時間で積分値 (N_zeta_arr, cov_arr) を返す
    """
    zeta_min = zeta_0 - step_rad_pm
    zeta_max = zeta_0 + step_rad_pm

    N_zeta_arr = np.full(nT, np.nan, dtype=float)
    cov_arr    = np.zeros(nT, dtype=float)

    for count_t, t0 in enumerate(t0_list):
        tmin = t0 - np.timedelta64(int(win_sec), 's')
        tmax = t0 + np.timedelta64(int(win_sec), 's')

        alpha_, zeta_, flux_ = _gather_samples(energy_i, tmin, tmax, 0., 180., 0., 360.)
        if alpha_ is None:
            continue

        tri, Z, A, F = build_triangulation(alpha_, zeta_, flux_)
        if tri is None:
            continue

        N_, cov_ = integrate_sinalpha_on_rect(
            tri, Z, A, F,
            alpha_min, alpha_max,
            zeta_min, zeta_max,
            n_samples=20000
        )

        N_zeta_arr[count_t] = N_ * 1E2 * np.sqrt(5.) * np.sqrt(proton_mass / elementary_charge) * energy_width[energy_i - 1] / np.sqrt(energy_ax[energy_i])
        cov_arr[count_t] = cov_

    da_N = xr.DataArray(
        N_zeta_arr,
        dims=['time'],
        coords={'time': t0_list_ns,
                'zeta_center': zeta_0,
                'energy_center_keV': energy_ax[energy_i]}
    )
    da_cov = xr.DataArray(
        cov_arr,
        dims=['time'],
        coords={'time': t0_list_ns,
                'zeta_center': zeta_0,
                'energy_center_keV': energy_ax[energy_i]}
    )
    return zeta_0, da_N, da_cov

# メインループ側
for energy_i in range(energy_num):
    if energy_i == 0:
        continue

    results = Parallel(n_jobs=os.cpu_count())(  # CPU コア数に応じて変える
        delayed(compute_single_zeta)(energy_i, z0)
        for z0 in zeta_0_list
    )

    N_da_list  = []
    cov_da_list = []
    for z0, da_N, da_cov in results:
        N_da_list.append(da_N)
        cov_da_list.append(da_cov)

    N_all = xr.concat(N_da_list, dim='zeta_center')
    cov_all = xr.concat(cov_da_list, dim='zeta_center')

    N_all.name = f'N_per_energy_{energy_i}'
    cov_all.name = f'N_per_energy_cov_{energy_i}'

    N_all.attrs['units'] = '[s m^{-3}]'

    N_per_energy_list[energy_i] = N_all
    N_per_energy_cov_list[energy_i] = cov_all

    print(f'energy_i={energy_i} done')


In [ ]:
N_per_energy_list[5]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

time_ax_range_min_lim = np.datetime64('2017-11-15T16:17:00')
time_ax_range_max_lim = np.datetime64('2017-11-15T16:22:00')

energy_i    = 5

gs  = plt.figure(figsize=(10, 6)).add_gridspec(2, 20, hspace=0.05)
ax0 = plt.gcf().add_subplot(gs[0, :19])
ax1 = plt.gcf().add_subplot(gs[1, :19], sharex=ax0)
cax = plt.gcf().add_subplot(gs[:, 19])

ax0.plot(da_B_256Hz_perp_bandpass_amp.time,
         da_B_256Hz_perp_bandpass_amp.data, lw=0.5, c='k')
ax0.set_ylabel(r'$|\mathbf{B}_{\mathrm{w}}|$' + '\n[nT]')
ax0.set_title(f'energy = {energy_ax[energy_i]:.4f} keV, '
              + r'$125\degree < \alpha < 145\degree$')
ax0.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax0.minorticks_on(); ax0.grid(True, which='both', linestyle='--', alpha=0.5)
ax0.set_ylim(0, 10); ax0.set_yticks(np.arange(0, 10.1, 2))
ax0.set_xlim(time_ax_range_min_lim, time_ax_range_max_lim)
ax0.tick_params(labelbottom=False)

N_da    = N_per_energy_list[energy_i].sel(time=slice(time_ax_range_min_lim, time_ax_range_max_lim))
time    = N_da['time'].values           # (151,)
zeta    = N_da['zeta_center'].values    # (12,)
N       = N_da.values                   # (zeta, time)

N_norm  = N / np.nanmax(N)

dz          = np.diff(zeta).mean()
zeta_edges  = np.concatenate([[zeta[0] - dz/2], (zeta[:-1] + zeta[1:]) / 2, [zeta[-1] + dz/2]])

dt          = np.diff(time).astype('timedelta64[s]').astype(float).mean()
time_edges  = np.concatenate([[time[0] - np.timedelta64(int(dt/2),'s')], time + np.timedelta64(int(dt/2),'s')])

pcm         = ax1.pcolormesh(time_edges, zeta_edges, N_norm, cmap='jet', shading='auto', vmin=0.6, vmax=1)
valid_t     = np.any(np.isfinite(N_norm), axis=0)
idx         = np.empty(time.size, dtype=int)
idx[valid_t] = np.nanargmax(N_norm[:, valid_t], axis=0)
ze_peak     = zeta[idx[valid_t]]
ax1.scatter(time[valid_t], ze_peak, marker='x', s=30, c='k', lw=0.8, zorder=3)

ax1.set_ylabel(r'$\zeta$' + '\n[deg]')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
ax1.minorticks_on(); ax1.grid(True, which='both', linestyle='--', alpha=0.5)
ax1.set_ylim(0, 360); ax1.set_yticks(np.arange(0, 361, 45))
ax1.set_xlim(time_ax_range_min_lim, time_ax_range_max_lim)

plt.colorbar(pcm, cax=cax)
plt.tight_layout()
plt.show()